<a href="https://colab.research.google.com/github/Nagesh-1906/Nagesh-1906/blob/main/BDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install PySpark
In this step, we install Apache Spark's Python API (`pyspark`) in order to process large amounts of listening data efficiently.

In [ ]:
!pip install pyspark

## Step 2: Initialize Spark Session and Load Data
Here, we spin up a local Spark session and read our cleaned Spotify listening history. We enable automatic schema inference and ensure double-quoted fields are parsed correctly.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MusicRecommendation").getOrCreate()

music_df = spark.read.option("header", True) \
    .option("quote", '"').option("escape", '"') \
    .option("inferSchema", True).csv("spotify_cleaned.csv")

music_df.show(5, truncate=False)

+-------------------+-------------+------------------------------------------------+------------------------------------------------+---------+-------+
|timestamp          |artist       |song                                            |album                                           |ms_played|skipped|
+-------------------+-------------+------------------------------------------------+------------------------------------------------+---------+-------+
|2024-05-28 12:35:14|Arijit Singh |Satranga (From "ANIMAL")                        |Satranga (From "ANIMAL")                        |117653   |true   |
|2024-05-28 12:35:18|Ram Sampath  |Sajni (From "Laapataa Ladies")                  |Sajni (From "Laapataa Ladies")                  |2810     |true   |
|2024-05-28 12:35:33|Mohammad Faiz|Dekhha Tenu (From "Mr. And Mrs. Mahi")          |Dekhha Tenu                                     |14535    |true   |
|2024-05-28 12:35:43|Badshah      |Soulmate                                        |Ek T

## Step 3: Enforce Correct Data Types
To prevent numerical overflow or analytical errors, we explicitly cast the listening duration column `ms_played` to a 64-bit long integer.

In [ ]:
from pyspark.sql import functions as F

# Explicitly convert playback duration to a number
music_df = music_df.withColumn(
    "ms_played",
    F.col("ms_played").cast("long")
)

music_df.select("artist", "song", "ms_played").show(5, truncate=False)

+-------------+------------------------------------------------+---------+
|artist       |song                                            |ms_played|
+-------------+------------------------------------------------+---------+
|Arijit Singh |Satranga (From "ANIMAL")                        |117653   |
|Ram Sampath  |Sajni (From "Laapataa Ladies")                  |2810     |
|Mohammad Faiz|Dekhha Tenu (From "Mr. And Mrs. Mahi")          |14535    |
|Badshah      |Soulmate                                        |9717     |
|Sachin-Jigar |Tum Se (From "Teri Baaton Mein Aisa Uljha Jiya")|3039     |
+-------------+------------------------------------------------+---------+
only showing top 5 rows


## Step 4: Verify Data Integrity
Before analyzing, we inspect the total row count, print the inferred Spark schema, and check for any unexpected missing (null) values.

In [ ]:
print("Total records:", music_df.count())

music_df.printSchema()

print("Missing values:")
from pyspark.sql.functions import col, sum

expressions = []

for c in music_df.columns:
    expression = sum(
        col(c).isNull().cast("int")
    ).alias(c)

    expressions.append(expression)

music_df.select(expressions).show()

Total records: 15114
root
 |-- timestamp: timestamp (nullable = true)
 |-- artist: string (nullable = true)
 |-- song: string (nullable = true)
 |-- album: string (nullable = true)
 |-- ms_played: integer (nullable = true)
 |-- skipped: boolean (nullable = true)

Missing values:
+---------+------+----+-----+---------+-------+
|timestamp|artist|song|album|ms_played|skipped|
+---------+------+----+-----+---------+-------+
|        0|     0|   0|    0|        0|      0|
+---------+------+----+-----+---------+-------+



## Step 5: Identify Most-Played Artists
We aggregate the listening dataset by artist to discover who you listen to most frequently.

In [ ]:
from pyspark.sql.functions import desc

top_artists = music_df.groupBy("artist").count().orderBy(desc("count"))

top_artists.show(10, truncate=False)

+-------------------+-----+
|artist             |count|
+-------------------+-----+
|Anirudh Ravichander|2460 |
|Pritam             |455  |
|Santhosh Narayanan |372  |
|Sid Sriram         |276  |
|A.R. Rahman        |266  |
|Devi Sri Prasad    |225  |
|Thaman S           |205  |
|Jakes Bejoy        |200  |
|Arijit Singh       |199  |
|Sushin Shyam       |185  |
+-------------------+-----+
only showing top 10 rows


## Step 6: Identify Most-Played Songs
Next, we aggregate the listening dataset by unique artist-song combinations to identify your top individual tracks.

In [ ]:
top_songs = music_df.groupBy("artist", "song").count().orderBy(desc("count"))

top_songs.show(10, truncate=False)

+-------------------+----------------------------------+-----+
|artist             |song                              |count|
+-------------------+----------------------------------+-----+
|Leon James         |Gundellonaa                       |87   |
|Imagine Dragons    |Bones                             |71   |
|Anirudh Ravichander|Gaali Vaaluga                     |65   |
|Sunny M.R.         |Anuvanuvuu                        |65   |
|Anirudh Ravichander|Master the Blaster (From "Master")|64   |
|Darshan Raval      |Mahiye Jinna Sohna                |64   |
|Sai Abhyankkar     |Katchi Sera - From "Think Indie"  |61   |
|Anirudh Ravichander|Baitikochi Chuste                 |61   |
|Anirudh Ravichander|Chaleya                           |60   |
|Anirudh Ravichander|Psycho Saiyaan (From "Saaho")     |59   |
+-------------------+----------------------------------+-----+
only showing top 10 rows


In [ ]:
music_df.select(
    (F.sum("ms_played") / 60000).alias("total_minutes"),
    (F.avg("ms_played") / 60000).alias("average_minutes"),
    (F.sum("ms_played")/60000/60).alias("total_hours"),
).show()

+-----------------+-----------------+-----------------+
|    total_minutes|  average_minutes|      total_hours|
+-----------------+-----------------+-----------------+
|8983.211616666667|0.594363611000838|149.7201936111111|
+-----------------+-----------------+-----------------+



In [ ]:
music_df.filter(F.col("skipped") == "True").select("artist", "song", "ms_played") \
    .show(10, truncate=False)

+---------------+---------------------------------------------------------+---------+
|artist         |song                                                     |ms_played|
+---------------+---------------------------------------------------------+---------+
|Arijit Singh   |Satranga (From "ANIMAL")                                 |117653   |
|Ram Sampath    |Sajni (From "Laapataa Ladies")                           |2810     |
|Mohammad Faiz  |Dekhha Tenu (From "Mr. And Mrs. Mahi")                   |14535    |
|Badshah        |Soulmate                                                 |9717     |
|Sachin-Jigar   |Tum Se (From "Teri Baaton Mein Aisa Uljha Jiya")         |3039     |
|Diljit Dosanjh |Naina (From "Crew")                                      |4864     |
|Darshan Raval  |Tu Hai                                                   |3188     |
|Tanishk Bagchi |Agar Ho Tum (From "Mr. And Mrs. Mahi")                   |53160    |
|Faheem Abdullah|Ishq - From "Lost;Found"             

In [ ]:
music_df.groupBy("artist", "song") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)

+-------------------+----------------------------------+-----+
|artist             |song                              |count|
+-------------------+----------------------------------+-----+
|Leon James         |Gundellonaa                       |87   |
|Imagine Dragons    |Bones                             |71   |
|Anirudh Ravichander|Gaali Vaaluga                     |65   |
|Sunny M.R.         |Anuvanuvuu                        |65   |
|Anirudh Ravichander|Master the Blaster (From "Master")|64   |
|Darshan Raval      |Mahiye Jinna Sohna                |64   |
|Sai Abhyankkar     |Katchi Sera - From "Think Indie"  |61   |
|Anirudh Ravichander|Baitikochi Chuste                 |61   |
|Anirudh Ravichander|Chaleya                           |60   |
|Anirudh Ravichander|Psycho Saiyaan (From "Saaho")     |59   |
+-------------------+----------------------------------+-----+
only showing top 10 rows


## Step 7: Build a General Recommendation Score
We define a personalized preference score for all tracks in the history. The score weights the total listening duration and overall play count positively, while applying a penalty for each time the track was skipped:

$$\text{Score} = (\text{Plays} \times 2) + \left(\frac{\text{Total Play Minutes}}{10}\right) - (\text{Skips} \times 2)$$

In [ ]:

# Calculate song statistics
recommendations = music_df.groupBy("artist", "song").agg(
    F.count("*").alias("play_count"),
    F.sum("ms_played").alias("total_ms_played"),
    F.avg("ms_played").alias("avg_ms_played"),
    F.sum(
        F.when(F.col("skipped") == "True", 1).otherwise(0)
    ).alias("skip_count")
)
recommendations.show(10,truncate=False)

# Calculate a simple preference score
recommendations = recommendations.withColumn(
    "score",
    F.col("play_count") * 2 +
    (F.col("total_ms_played") / 60000) / 10 -
    F.col("skip_count") * 2
)

# Display top recommended songs
recommendations.orderBy(
    F.desc("score")
).select(
    "artist", "song", "play_count", "skip_count", "score"
).show(20, truncate=False)

+-------------------+------------------------------------------------+----------+---------------+-----------------+----------+
|artist             |song                                            |play_count|total_ms_played|avg_ms_played    |skip_count|
+-------------------+------------------------------------------------+----------+---------------+-----------------+----------+
|Anirudh Ravichander|Naa Ready (From "Leo")                          |6         |572842         |95473.66666666667|4         |
|Sagar              |Mee Intiki Mundhu                               |25        |1747497        |69899.88         |18        |
|Anuv Jain          |Baarishein                                      |10        |47063          |4706.3           |7         |
|G. V. Prakash      |Sai Pallavi's Intro (From "Amaran")             |1         |1194           |1194.0           |1         |
|Anurag Kulkarni    |Fire Song (From "Kanguva") (Telugu)             |2         |9692           |4846.0        

## Step 8: Refine Recommendations Using Favorite Artists
To further improve recommendation relevance, we first locate your top 10 artists by aggregate play volume and time. Then, we extract candidate songs exclusively belonging to those artists, ranking them using our play-to-skip preference formula.

In [ ]:

# 1. Calculate artist preferences
artist_scores = music_df.groupBy("artist").agg(
    F.count("*").alias("plays"),
    F.sum("ms_played").alias("total_ms")
).withColumn(
    "artist_score",
    F.col("plays") + F.col("total_ms") / 60000 / 10
)

# 2. Select the top 10 favorite artists
favorite_artists = artist_scores.orderBy(
    F.desc("artist_score")
).limit(10).select("artist")

# 3. Find songs by favorite artists
candidate_songs = music_df.join(
    favorite_artists, on="artist", how="inner"
)

# 4. Rank songs by listening duration
ranked_songs = candidate_songs.groupBy(
    "artist", "song"
).agg(
    F.count("*").alias("plays"),
    F.sum("ms_played").alias("total_ms"),
    F.sum(
        F.when(F.col("skipped") == "True", 1).otherwise(0)
    ).alias("skips")
).withColumn(
    "score",
    F.col("total_ms") / 60000
    + F.col("plays") * 2
    - F.col("skips") * 2
)

# 5. Show 20 ranked song recommendations
ranked_songs.orderBy(
    F.desc("score")
).select(
    "artist", "song", "plays", "score"
).show(20, truncate=False)


+-------------------+---------------------------------------------------------------------------------------------+-----+------------------+
|artist             |song                                                                                         |plays|score             |
+-------------------+---------------------------------------------------------------------------------------------+-----+------------------+
|Anirudh Ravichander|Baitikochi Chuste                                                                            |61   |234.5238          |
|Anirudh Ravichander|Gaali Vaaluga                                                                                |65   |208.76263333333333|
|Arijit Singh       |Bekhayali (Arijit Singh Version)                                                             |36   |124.56221666666667|
|Anirudh Ravichander|Hoyna Hoyna                                                                                  |52   |111.46809999999999|
|Arijit Singh